<a href="https://colab.research.google.com/github/1HPz/Super-AI-Engineer-Season-6/blob/main/Typhoon_fahmai_rag_challenge_1_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 RAG Challenge — Level 1

**Pipeline:**
1. โหลด Knowledge Base จาก `.md` files
2. Chunk + Embed ด้วย `BGE-M3`
3. Retrieve ด้วย FAISS
4. Generate คำตอบด้วย `OpenThaiGPT`
5. Export `submission.csv`

> ⚠️ ก่อนรัน: **Runtime → Change runtime type → T4 GPU** แนะนำครับบ

## 📦 Cell 1: ติดตั้ง Libraries

In [ ]:
!pip install transformers accelerate sentence-transformers faiss-cpu pandas -q

## 📁 Cell 2: Upload และ Unzip ไฟล์

In [ ]:
from google.colab import files
import zipfile, os

# Upload .zip file
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Unzip
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# ตรวจสอบ structure
for root, dirs, files_ in os.walk('/content/data'):
    level = root.replace('/content/data', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_[:3]:
            print(f'{subindent}{f}')
        if len(files_) > 3:
            print(f'{subindent}... ({len(files_)} files total)')

## 📚 Cell 3: โหลด Knowledge Base

In [ ]:
import glob

KB_PATH = '/content/data/knowledge_base'

def load_knowledge_base(kb_path):
    docs = []
    for filepath in sorted(glob.glob(f'{kb_path}/**/*.md', recursive=True)):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        # ดึง category จาก path
        parts = filepath.replace(kb_path + '/', '').split('/')
        category = parts[0]  # policies / products / store_info
        filename = os.path.basename(filepath)
        docs.append({
            'filename': filename,
            'category': category,
            'content': content,
            'filepath': filepath
        })
    print(f'✅ โหลดได้ {len(docs)} ไฟล์')
    return docs

docs = load_knowledge_base(KB_PATH)

# สรุป
from collections import Counter
cats = Counter(d['category'] for d in docs)
for cat, count in cats.items():
    print(f'  {cat}: {count} ไฟล์')

## ✂️ Cell 4: Chunk Documents

In [ ]:
def chunk_document(doc, chunk_size=800, overlap=100):
    """
    แบ่ง document เป็น chunk เล็กๆ
    - chunk_size: จำนวนตัวอักษรต่อ chunk
    - overlap: จำนวนตัวอักษรที่ overlap ระหว่าง chunk
    """
    content = doc['content']
    chunks = []
    start = 0
    while start < len(content):
        end = start + chunk_size
        chunk_text = content[start:end]
        chunks.append({
            'text': chunk_text,
            'filename': doc['filename'],
            'category': doc['category'],
            'chunk_id': len(chunks)
        })
        start += chunk_size - overlap
    return chunks

# Chunk ทุกไฟล์
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_document(doc))

print(f'✅ แบ่งได้ {len(all_chunks)} chunks จาก {len(docs)} ไฟล์')
print(f'   เฉลี่ย {len(all_chunks)/len(docs):.1f} chunks ต่อไฟล์')

## 🔢 Cell 5: Embed ด้วย BGE-M3

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

print('กำลังโหลด embedding model...')
embedder = SentenceTransformer('BAAI/bge-m3')

texts = [c['text'] for c in all_chunks]

print(f'กำลัง embed {len(texts)} chunks...')
embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    batch_size=16,
    normalize_embeddings=True  # สำหรับ cosine similarity
)

print(f'✅ Embedding shape: {embeddings.shape}')

## 🗄️ Cell 6: สร้าง FAISS Index

In [ ]:
import faiss

dim = embeddings.shape[1]

# ใช้ Inner Product (เหมาะกับ normalized embeddings = cosine similarity)
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings, dtype='float32'))

print(f'✅ FAISS index พร้อมแล้ว ({index.ntotal} vectors)')

def retrieve(query, top_k=5):
    """ค้นหา chunk ที่เกี่ยวข้องกับ query"""
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype='float32'), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text': all_chunks[idx]['text'],
            'filename': all_chunks[idx]['filename'],
            'score': float(score)
        })
    return results

# ทดสอบ
test = retrieve('Watch S3 Ultra กันน้ำ ATM')
print('\n🔍 ทดสอบ retrieve:')
print(f'  → {test[0]["filename"]} (score: {test[0]["score"]:.3f})')
print(f'  → {test[0]["text"][:200]}...')

## 🤖 Cell 7: โหลด Typhoon

In [ ]:
import requests
import json
import re
import pandas as pd
from tqdm.auto import tqdm

# --- CONFIG สำหรับ Typhoon ตาม cURL ---
API_URL = "http://thaillm.or.th/api/typhoon/v1/chat/completions"
API_KEY = "ThaiLLM"
MODEL_NAME_INTERNAL = "/model"

print(f'✅ เตรียมเชื่อมต่อ Typhoon API: {API_URL}')

## 🧠 Cell 8: RAG Pipeline

In [ ]:
def build_prompt(question, choices, context_chunks):
    """สร้าง prompt โดยใช้ XML Tag เพื่อแยกส่วนข้อมูลให้ AI เข้าใจง่ายขึ้น"""

    # 1. เตรียมส่วนข้อมูลอ้างอิงในรูปแบบ XML
    context_xml = ""
    for i, c in enumerate(context_chunks):
        context_xml += f'<reference id="{i+1}" source="{c["filename"]}">\n{c["text"]}\n</reference>\n'

    # 2. เตรียมส่วนตัวเลือกในรูปแบบ XML
    choices_xml = ""
    for i, choice in enumerate(choices):
        choices_xml += f'<option id="{i+1}">{choice}</option>\n'

    # 3. ประกอบร่าง Prompt ทั้งหมด
    prompt = f"""<system_instruction>
คุณเป็น AI ผู้ช่วยของร้านฟ้าใหม่ หน้าที่ของคุณคือวิเคราะห์ข้อมูลใน <context> เพื่อตอบคำถามใน <question>
กฎการตอบ:
- เลือกคำตอบที่ถูกต้องที่สุดจาก <options> เท่านั้น
- ตอบเป็น "ตัวเลข ID" (1-10) เพียงอย่างเดียว
- ห้ามเขียนคำอธิบาย หรือ Tag อื่นๆ ในคำตอบสุดท้าย
</system_instruction>

<context>
{context_xml}</context>

<task>
    <question>{question}</question>
    <options>
{choices_xml}    </options>
</task>

คำตอบ (หมายเลข ID):"""
    return prompt

def extract_answer(text):
    """ดึงเฉพาะตัวเลข ID ออกจากคำตอบ"""
    text = text.strip()
    # ค้นหาตัวเลข 1-10
    match = re.search(r'\b(10|[1-9])\b', text)
    if match:
        return int(match.group(1))
    return 1

def answer_question(question, choices, top_k=5):
    """ฟังก์ชันหลักสำหรับส่งคำถามผ่าน API"""
    # Retrieve ข้อมูล
    context_chunks = retrieve(question, top_k=top_k)

    # สร้าง XML Prompt
    prompt = build_prompt(question, choices, context_chunks)

    headers = {
        "Content-Type": "application/json",
        "apikey": API_KEY
    }
    data = {
        "model": MODEL_NAME_INTERNAL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 10,
        "temperature": 0.3 # ค่าเดิมตาม cURL
    }

    try:
        response = requests.post(API_URL, headers=headers, json=data, timeout=60)
        response.raise_for_status()
        generated = response.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f"⚠️ API Error: {e}")
        generated = ""

    return extract_answer(generated), generated.strip()

# --- ทดสอบระบบ ---
test_q = 'Watch S3 Ultra กันน้ำได้กี่ ATM ครับ'
test_choices = ['3 ATM', 'IP68', '5 ATM', 'IP67', '10 ATM', '20 ATM', 'IPX8', '1 ATM', 'ไม่มีข้อมูล', 'ไม่เกี่ยวข้อง']
ans, raw = answer_question(test_q, test_choices)
print(f"\n🔍 ผลทดสอบ XML Tag:")
print(f"AI Response: '{raw}'")
print(f"สรุปเลขข้อ: {ans} ({test_choices[ans-1]})")

## 🚀 Cell 9: รันทุกคำถาม

In [ ]:
import pandas as pd
from tqdm import tqdm

# โหลด questions
df = pd.read_csv('/content/data/questions.csv')
choice_cols = [f'choice_{i}' for i in range(1, 11)]

print(f'จำนวนคำถาม: {len(df)} ข้อ')
print('เริ่มรัน RAG pipeline...\n')

results = []
errors = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Answering'):
    qid = row['id']
    question = row['question']
    choices = [str(row[c]) for c in choice_cols]

    try:
        ans, raw = answer_question(question, choices)
        results.append({'id': qid, 'answer': ans})
    except Exception as e:
        print(f'\n❌ Q{qid} error: {e}')
        results.append({'id': qid, 'answer': 1})  # fallback
        errors.append(qid)

print(f'\n✅ ตอบครบ {len(results)} ข้อ')
if errors:
    print(f'⚠️  มี error {len(errors)} ข้อ: {errors}')

## 💾 Cell 10: Export submission.csv

In [ ]:
# สร้าง submission
submission = pd.DataFrame(results)
submission = submission.sort_values('id').reset_index(drop=True)

# ตรวจสอบ
print('ตัวอย่าง submission:')
print(submission.head(10))
print(f'\nการกระจายคำตอบ:')
print(submission['answer'].value_counts().sort_index())

# บันทึก
OUTPUT_PATH = '/content/submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)
print(f'\n✅ บันทึกที่ {OUTPUT_PATH}')

# Download อัตโนมัติ
from google.colab import files
files.download(OUTPUT_PATH)